# 04 — Round Persistence and ProofFrame Diff

Two routemap-window capabilities that turn a sequence of capability calls into a durable, comparable audit trail:

1. **Round Persistence** (Batch 6) — capture a sequence of capability results inside an audit package via the external recorder API: `start_round(...)`, `record_round_event(...)`, `finalize_round(...)`. The recorder writes `audit/round_events.jsonl` atomically (tempfile + `os.replace`) at finalize time.
2. **ProofFrame Diff** (Batch 7) — given two finalized rounds, `AuditQuery.diff_proof_frames(round_a, round_b)` derives a per-frame delta over their `proof_frame_result` events. Frame identity is `(support_digest, binding_items)`; per-atom identity is `condition_key` within the same `support_digest`.

**Capability runtimes do *not* import `factgraph.audit`.** The caller invokes a capability, then explicitly records the result outside the runtime — preserving the application↛audit one-direction invariant.

**First-slice persisted event kinds:**

- lifecycle: `round_started`, `round_finalized`
- capability: `check_result`, `diagnose_result`, `fact_overlay_result`, `why_not_result`, `proof_frame_result`

**Prerequisites:** [03_proofframe_rule_overlays.ipynb](03_proofframe_rule_overlays.ipynb). This is the final chapter; the integrated walkthrough is `examples/round_story_full_demo.py` (run as a script for end-to-end smoke verification).

## Setup + run prerequisite phases

The round-recorder events project from existing capability call results. We rebuild the chapter-1/2/3 fixture inline, then run Q1 Check / Q2 Diagnose / Q3 Fact Overlay / Q4 Why-not / ProofFrame baseline + overlay to obtain the request/result pairs the recorder will project from.

In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import dataclass
from pathlib import Path
from tempfile import TemporaryDirectory

_repo_root = Path.cwd()
if not (_repo_root / 'src').exists() and (_repo_root.parent / 'src').exists():
    _repo_root = _repo_root.parent
_src_dir = _repo_root / 'src'
if str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))

from factgraph.application import (
    build_fact_value_override,
    build_schema_index,
    build_why_not_candidate_universe,
    check_derivation_binding,
    check_fact_overlay_binding,
    check_why_not_universe,
    diagnose_derivation_binding,
    entity_info,
    field_predicate,
    recheck_proof_frame,
    resolve_selector,
)
from factgraph.application.protocol import (
    CheckRequest,
    CompiledDerivationPlan,
    CompiledHeadCall,
    DiagnoseRequest,
    EntitySelector,
    EvaluationOverlay,
    FactOverlayCheckRequest,
    FieldPath,
    ProofFrameRecheckRequest,
    WhyNotUniverseRequest,
)
from factgraph.audit import AuditQuery, load_audit_package
from factgraph.audit.round_events import (
    finalize_round,
    project_check_event_payload,
    project_diagnose_event_payload,
    project_fact_overlay_event_payload,
    project_proof_frame_event_payload,
    project_why_not_event_payload,
    record_round_event,
    start_round,
)
from factgraph.core.evidence.write_protocol import set_field
from factgraph.core.store import Store
from factgraph.core.store._support import SupportArtifact
from factgraph.sdk import Entity, Field, Identity, compile_schema_from_classes


class Person(Entity):
    name: str = Identity()
    age: int = Field()
    region: str = Field()


@dataclass(frozen=True)
class SeededPerson:
    e_ref: str


def seed(store, index, *, name, age, region):
    info = entity_info(index, 'Person')
    ref = resolve_selector(EntitySelector(entity_type='Person', identity={'name': name}), index=index)
    encoded = ref.encoded_ref or ''
    set_field(store.ledger, info.exists_predicate_id, encoded, [])
    set_field(store.ledger, info.identity_predicates['name'].pred_id, encoded, [('string', name)])
    set_field(store.ledger, field_predicate(index, 'Person', 'age').pred_id, encoded, [('int', age)])
    set_field(store.ledger, field_predicate(index, 'Person', 'region').pred_id, encoded, [('string', region)])
    return SeededPerson(e_ref=encoded)


def _binding(*items):
    return tuple(sorted(items, key=lambda i: i[0]))


schema_ir = compile_schema_from_classes([Person])
index = build_schema_index(schema_ir)
store = Store(schema_ir)
people = {
    'alice': seed(store, index, name='alice', age=25, region='us'),
    'bob':   seed(store, index, name='bob',   age=30, region='eu'),
    'carol': seed(store, index, name='carol', age=28, region='us'),
}
person_info = entity_info(index, 'Person')
plan = CompiledDerivationPlan(
    derivation_id='round-story-person-snapshot',
    version='1.0',
    body_ir=[
        ('pred', person_info.exists_predicate_id, ['$p']),
        ('pred', field_predicate(index, 'Person', 'age').pred_id,    ['$p', '$age']),
        ('pred', field_predicate(index, 'Person', 'region').pred_id, ['$p', '$region']),
    ],
    heads=(CompiledHeadCall(
        target_pred_id=person_info.exists_predicate_id,
        head_var_names=('$p', '$age', '$region'),
    ),),
)

alice = people['alice']
check_request = CheckRequest(
    plan=plan,
    binding=_binding(('$p', alice.e_ref), ('$age', 25), ('$region', 'us')),
    engine='native',
)
check_result = check_derivation_binding(check_request, store=store)
support_artifact = check_result.evidence_envelope.proof
assert isinstance(support_artifact, SupportArtifact)

diagnose_request = DiagnoseRequest(
    plan=plan,
    binding=_binding(('$p', alice.e_ref), ('$age', 99), ('$region', 'us')),
    engine='native',
)
diagnose_result = diagnose_derivation_binding(diagnose_request, store=store)

age_overlay = EvaluationOverlay(
    fact_actions=(
        build_fact_value_override(
            store, index, e_ref=alice.e_ref,
            field=FieldPath(entity_type='Person', field_name='age'),
            new_value=30, note='Alice turns 30',
        ),
    ),
)
fact_overlay_request = FactOverlayCheckRequest(
    plan=plan,
    binding=_binding(('$p', alice.e_ref), ('$age', 30), ('$region', 'us')),
    overlay=age_overlay,
    engine='native',
)
fact_overlay_result = check_fact_overlay_binding(fact_overlay_request, store=store)

why_not_request = WhyNotUniverseRequest(
    plan=plan,
    candidate_universe=build_why_not_candidate_universe(
        plan,
        (
            {'$p': alice.e_ref, '$age': 30, '$region': 'us'},
            {'$p': people['bob'].e_ref, '$age': 30, '$region': 'eu'},
            {'$p': people['carol'].e_ref, '$age': 30, '$region': 'us'},
        ),
    ),
    engine='native',
)
why_not_result = check_why_not_universe(why_not_request, store=store)

baseline_pf_request = ProofFrameRecheckRequest(
    support_artifact=support_artifact,
    overlay=EvaluationOverlay(),
)
baseline_pf_result = recheck_proof_frame(baseline_pf_request, store=store)

overlay_pf_request = ProofFrameRecheckRequest(
    support_artifact=support_artifact,
    overlay=age_overlay,
)
overlay_pf_result = recheck_proof_frame(overlay_pf_request, store=store)

print(f'check.status              : {check_result.status}')
print(f'diagnose.status           : {diagnose_result.status}')
print(f'fact_overlay.status       : {fact_overlay_result.status}')
print(f'why_not.status            : {why_not_result.status}')
print(f'baseline_proofframe.status: {baseline_pf_result.status}')
print(f'overlay_proofframe.status : {overlay_pf_result.status}')

## 1. Build a minimal audit package on disk

The recorder writes round events into an existing audit package directory. v0.2 audit packages have a fixed layout:

```
package/
  manifest.json
  audit/
    run_ledger.jsonl
    candidate_ledger.jsonl
    accept_write_ledger.jsonl
    accept_failed.jsonl
    mapping_resolution.json
    decision_log.jsonl
    round_events.jsonl     <-- written by the recorder at finalize_round time
```

For demo isolation we build the package inside a `TemporaryDirectory` (so the notebook cleans up after itself). The cell below defines a `_minimal_audit_package(...)` helper that lays down empty audit files + a manifest, then enters the temp dir context for the rest of the demo.

In [ ]:
def _minimal_audit_package(package_dir: Path) -> Path:
    audit_dir = package_dir / 'audit'
    audit_dir.mkdir(parents=True)
    audit_files = {
        'run_ledger':           'audit/run_ledger.jsonl',
        'candidate_ledger':     'audit/candidate_ledger.jsonl',
        'accept_write_ledger':  'audit/accept_write_ledger.jsonl',
        'accept_failed':        'audit/accept_failed.jsonl',
        'mapping_resolution':   'audit/mapping_resolution.json',
        'decision_log':         'audit/decision_log.jsonl',
    }
    for key, rel_path in audit_files.items():
        path = package_dir / rel_path
        path.write_text('{}' if key == 'mapping_resolution' else '', encoding='utf-8')
    manifest = {'package_kind': 'audit', 'paths': {'audit_files': audit_files}}
    (package_dir / 'manifest.json').write_text(
        json.dumps(manifest, sort_keys=True), encoding='utf-8',
    )
    return package_dir


_tmp_ctx = TemporaryDirectory()
package_dir = _minimal_audit_package(Path(_tmp_ctx.name) / 'audit_pkg')
print(f'package_dir : {package_dir}')
print(f'manifest    : {(package_dir / "manifest.json").read_text()}')

## 2. Round 1 (baseline) — record 5 capability events and finalize

`start_round(round_id, event_ts=...)` returns an in-memory `RoundRecorder`. `record_round_event(recorder, kind=..., payload=..., event_ts=...)` appends one event in memory; nothing is written to disk yet. `finalize_round(recorder, package_dir, event_ts=...)` performs the atomic write (tempfile + `os.replace`) of the full sequence — `round_started` lifecycle marker, all recorded events in order, `round_finalized` lifecycle marker.

Each capability ships a `project_<kind>_event_payload(request, result)` helper that constructs the event payload — capability runtimes never know about the audit layer, but the projection helpers know how to encode their request / result into the durable shape.

In [ ]:
baseline_recorder = start_round('round-baseline', event_ts=100)

baseline_events = [
    ('check_result',        project_check_event_payload(check_request, check_result)),
    ('diagnose_result',     project_diagnose_event_payload(diagnose_request, diagnose_result)),
    ('fact_overlay_result', project_fact_overlay_event_payload(fact_overlay_request, fact_overlay_result)),
    ('why_not_result',      project_why_not_event_payload(why_not_request, why_not_result)),
    ('proof_frame_result',  project_proof_frame_event_payload(baseline_pf_request, baseline_pf_result)),
]
for offset, (kind, payload) in enumerate(baseline_events, start=1):
    record_round_event(baseline_recorder, kind=kind, payload=payload, event_ts=100 + offset)

finalize_round(baseline_recorder, package_dir, event_ts=200)

events_path = package_dir / 'audit' / 'round_events.jsonl'
print(f'round_events.jsonl exists : {events_path.exists()}')
print(f'rows after baseline       : {sum(1 for _ in events_path.open())}')

## 3. Round 2 (overlay) — record one ProofFrame event and finalize

Round 2 is intentionally minimal: one `proof_frame_result` event from the *overlay* recheck (the one that returned `invalidated`). This is the second of the two finalized rounds the diff will compare.

In [ ]:
overlay_recorder = start_round('round-overlay', event_ts=300)
record_round_event(
    overlay_recorder,
    kind='proof_frame_result',
    payload=project_proof_frame_event_payload(overlay_pf_request, overlay_pf_result),
    event_ts=301,
)
finalize_round(overlay_recorder, package_dir, event_ts=302)

print(f'rows after overlay round  : {sum(1 for _ in events_path.open())}')

## 4. Inspect the persisted JSONL rows directly

`round_events.jsonl` is plain newline-delimited JSON. Each row carries `round_id`, `kind`, `event_ts`, plus the projected `payload`. The cell below reads the file directly so the persisted shape is visible (we look at the first 4 rows: the baseline round_started lifecycle row and the first three capability events).

In [ ]:
raw_rows = [json.loads(line) for line in events_path.open()]
print(f'total rows : {len(raw_rows)}')
print()
for row in raw_rows[:4]:
    print(f'round_id={row["round_id"]!r:18}  kind={row["kind"]!r:24}  event_ts={row["event_ts"]}')

## 5. Load the package and inspect via `AuditQuery`

`load_audit_package(package_dir)` returns an `AuditPackage` with the full set of typed views over the audit files; `round_events` is one of them. `AuditQuery(package)` wraps it with query helpers — `list_rounds()`, `get_round_summary(round_id)`, `diff_proof_frames(...)`, etc.

In [ ]:
package = load_audit_package(package_dir)
query = AuditQuery(package)

print(f'rounds in package : {sorted(query.list_rounds())}')
for round_id in sorted(query.list_rounds()):
    summary = query.get_round_summary(round_id)
    print(
        f'  {round_id}: is_finalized={summary.is_finalized} '
        f'event_count={summary.event_count} '
        f'kinds={sorted(summary.kind_counts.items())}'
    )

## 6. ProofFrame Diff (Batch 7) — compare two finalized rounds

`AuditQuery.diff_proof_frames(round_a, round_b, *, include_partial=False, include_unchanged=False)` consumes only `proof_frame_result` rows. Frame identity is `(payload.request.support_digest, payload.result.binding_items)`; per-atom identity is `condition_key` scoped to the same `support_digest`.

Both rounds carry exactly one frame for the same `(support_digest, binding_items)` pair (Alice's age=25 derivation), but the baseline frame is `still_valid` and the overlay frame is `invalidated` — so we expect exactly one `FrameDelta` reporting a `frame_status_change` of `still_valid → invalidated`.

In [ ]:
diff = query.diff_proof_frames('round-baseline', 'round-overlay')

assert len(diff.frame_deltas) == 1, diff
delta = diff.frame_deltas[0]
assert delta.frame_status_change is not None
assert delta.frame_status_change.before == 'still_valid'
assert delta.frame_status_change.after  == 'invalidated'

print(f'frame_deltas count : {len(diff.frame_deltas)}')
print(f'  frame.support_digest      : {delta.frame_identity.support_digest!r}')
print(f'  frame.binding_items       : {delta.frame_identity.binding_items}')
print(f'  frame_status_change.before: {delta.frame_status_change.before}')
print(f'  frame_status_change.after : {delta.frame_status_change.after}')
print(f'  per-atom deltas           : {len(delta.atom_deltas)}')
for atom_delta in delta.atom_deltas:
    print(
        f'    atom {atom_delta.condition_key}: kind={atom_delta.kind} '
        f'verdict {atom_delta.before_verdict} -> {atom_delta.after_verdict}'
    )

In [ ]:
# Tear down the temporary audit package now that the demo is complete.
_tmp_ctx.cleanup()
print('temporary audit package removed.')

## Wrap-up

| Capability | API | Module |
|---|---|---|
| Round Recorder (Batch 6) | `start_round(...)` / `record_round_event(...)` / `finalize_round(...)` | `factgraph.audit.round_events` |
| Round inspection         | `load_audit_package(...)` + `AuditQuery.list_rounds() / get_round_summary(...)` | `factgraph.audit` |
| ProofFrame Diff (Batch 7)| `AuditQuery.diff_proof_frames(round_a, round_b)`                          | `factgraph.audit.proof_frame_diff` |

**Lenient reader:** `AuditQuery` rejects partial rounds (no `round_finalized` marker) by default — opt in with `include_partial=True`. Future-versioned rows (`schema_version >= 2.0`) are skipped with a `DIFF_FUTURE_KIND_SKIPPED` warning. Frames with empty `atom_verdicts` are marked `rule_refs_unsupported` and produce no per-atom delta.

**Public-surface boundary (Batch 8):** Both Round Persistence and ProofFrame Diff are advanced-importable (`factgraph.audit.round_events` and `factgraph.audit.proof_frame_diff`); v0.2 ships **no** SDK shells or service routes for them.

**Series complete.** The integrated walkthrough is `examples/round_story_full_demo.py` — run as `python examples/round_story_full_demo.py` for an end-to-end smoke that exercises every capability covered across chapters 1–4 against the same fixture, and asserts on the full `EXPECTED_PHASE_SUMMARY` contract.